In [ ]:
!pip install timm transformers datasets accelerate --quiet

In [ ]:
import zipfile
import os
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
zip_path="/content/drive/MyDrive/Research_project/TrashNeXt Dataset.zip"
extract_path = "/content"


In [ ]:
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted to:", extract_path)
!ls {extract_path}

Dataset extracted to: /content
 drive	 __MACOSX   sample_data  'TrashNeXt Dataset'


In [ ]:
import os
from PIL import Image
from tqdm import tqdm

def is_corrupted_image(file_path):
    """Check if an image file is corrupted."""
    try:
        with Image.open(file_path) as img:
            img.verify()  # Verify integrity
        return False
    except (IOError, SyntaxError, Image.DecompressionBombError):
        return True

def remove_corrupted_images(dataset_path):
    """Scan and remove corrupted images + hidden macOS files."""
    corrupted_count = 0
    total_images = 0

    for root, _, files in os.walk(dataset_path):
        for file in tqdm(files, desc=f"Scanning {os.path.basename(root)}"):

            # Skip hidden files (.DS_Store, ._files)
            if file.startswith("."):
                file_path = os.path.join(root, file)
                try:
                    os.remove(file_path)
                    corrupted_count += 1
                    print(f"Removed hidden file: {file_path}")
                except:
                    pass
                continue

            # Check only image formats
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                total_images += 1
                file_path = os.path.join(root, file)
                if is_corrupted_image(file_path):
                    try:
                        os.remove(file_path)
                        corrupted_count += 1
                        print(f"Removed corrupted image: {file_path}")
                    except Exception as e:
                        print(f"Error removing {file_path}: {e}")

    print("\n✅ Scan completed!")
    print(f"📷 Total images scanned: {total_images}")
    print(f"🗑️ Corrupted/hidden files removed: {corrupted_count}")

# Run the cleaner
dataset_path = '/content/dataset'  # Your dataset path
remove_corrupted_images(dataset_path)


Scanning dataset: 100%|██████████| 1/1 [00:00<00:00, 3127.74it/s]


Removed hidden file: /content/dataset/.DS_Store


Scanning Valid: 100%|██████████| 1/1 [00:00<00:00, 7256.58it/s]


Removed hidden file: /content/dataset/Valid/.DS_Store


Scanning Train: 100%|██████████| 1/1 [00:00<00:00, 2592.28it/s]


Removed hidden file: /content/dataset/Train/.DS_Store


Scanning Test: 100%|██████████| 1/1 [00:00<00:00, 6636.56it/s]


Removed hidden file: /content/dataset/Test/.DS_Store


Scanning metal: 100%|██████████| 258/258 [00:00<00:00, 984.61it/s]


✅ Scan completed!
📷 Total images scanned: 23625
🗑️ Corrupted/hidden files removed: 4


In [ ]:
from PIL import Image
import os

def remove_truncated_images(dataset_path):
    for root, _, files in os.walk(dataset_path):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                try:
                    img_path = os.path.join(root, file)
                    img = Image.open(img_path)
                    img.verify()
                except Exception as e:
                    print(f"Removing corrupted/truncated: {img_path}")
                    os.remove(img_path)

remove_truncated_images("/content/dataset")


In [ ]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import timm
import numpy as np
import os
from tqdm import tqdm

In [ ]:
class HybridFeatureExtractor(nn.Module):
    def __init__(self):
        super(HybridFeatureExtractor, self).__init__()
        # EfficientNetV2 feature extractor
        self.efficientnet = models.efficientnet_v2_s(weights="IMAGENET1K_V1")
        self.efficientnet.classifier = nn.Identity()

        # LeViT feature extractor
        self.levit = timm.create_model('levit_128s', pretrained=True) # Changed from 'levit_256'

        # Replace classification heads with identity
        self.levit.head = nn.Identity()
        if hasattr(self.levit, 'dist_head'):
            self.levit.dist_head = nn.Identity()

        # ✅ Force LeViT to skip forward_head completely
        def forward_override(x):
            # run backbone (patch embedding + transformer encoder + pooling)
            x = self.levit.forward_features(x)   # [B, N, D]
            # take CLS token only (LeViT uses CLS for classification)
            x = x[:, 0, :]                       # [B, D]
            return x
        self.levit.forward = forward_override

    def forward(self, x):
        # Resize for each model
        x_eff = transforms.functional.resize(x, (300, 300))
        x_levit = transforms.functional.resize(x, (224, 224))

        with torch.no_grad():
            features_eff = self.efficientnet(x_eff)
            features_levit = self.levit(x_levit) # Changed from [B, 512] to [B, 384] for levit_128s

        return torch.cat((features_eff, features_levit), dim=1)

In [ ]:
def extract_features(data_loader, model, device):
    all_features = []
    all_labels = []

    model.eval()
    for images, labels in tqdm(data_loader, desc="Extracting features"):
        images = images.to(device)
        features = model(images)
        all_features.append(features.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    return np.concatenate(all_features), np.concatenate(all_labels)

In [ ]:
if __name__ == '__main__':
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # --- Data Loading ---
    # ❗️ IMPORTANT: Update these paths to your dataset location in Google Drive
    train_dir = "/content/dataset/Train" # Corrected path
    valid_dir = "/content/dataset/Valid" # Corrected path

    data_transforms = transforms.Compose([
        transforms.Resize((256, 256)), # Initial resize
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    train_dataset = datasets.ImageFolder(train_dir, transform=data_transforms)
    valid_dataset = datasets.ImageFolder(valid_dir, transform=data_transforms)

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False)
    valid_loader = DataLoader(valid_dataset, batch_size=16, shuffle=False)

    # --- Initialize model and extract ---
    feature_extractor = HybridFeatureExtractor().to(device)

    X_train, y_train = extract_features(train_loader, feature_extractor, device)
    X_valid, y_valid = extract_features(valid_loader, feature_extractor, device)

    print(f"Training features shape: {X_train.shape}")
    print(f"Validation features shape: {X_valid.shape}")

    # Save features for the next stage
    np.save('X_train.npy', X_train)
    np.save('y_train.npy', y_train)
    np.save('X_valid.npy', X_valid)
    np.save('y_valid.npy', y_valid)
    print("✅ Features saved!")

Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 194MB/s]


model.safetensors:   0%|          | 0.00/31.3M [00:00<?, ?B/s]

Extracting features:   0%|          | 1/1182 [00:01<25:22,  1.29s/it]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Extracting features: 100%|██████████| 148/148 [00:47<00:00,  3.10it/s]


Training features shape: (18898, 1664)
Validation features shape: (2363, 1664)
✅ Features saved!


In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import accuracy_score, log_loss
import matplotlib.pyplot as plt

# --- Load extracted features ---
X_train = np.load("X_train.npy")
y_train = np.load("y_train.npy")
X_valid = np.load("X_valid.npy")
y_valid = np.load("y_valid.npy")

In [ ]:
# Create DMatrix (optimized format for XGBoost)
dtrain = xgb.DMatrix(X_train, label=y_train)
dvalid = xgb.DMatrix(X_valid, label=y_valid)

# --- XGBoost Parameters ---
params = {
    'objective': 'multi:softprob',   # multi-class classification
    'num_class': len(np.unique(y_train)),
    'eval_metric': 'mlogloss',
    'learning_rate': 0.08884427064151632,
    'max_depth': 5,
    'subsample': 0.5500312962779268,
    'colsample_bytree': 0.5776303994727874,
    'gamma':0.44941288449556765,
    'seed': 42
}

In [ ]:
# --- Training with evaluation ---
num_rounds = 300   # total boosting rounds (like epochs)
evals = [(dtrain, 'train'), (dvalid, 'valid')]
evals_result = {}

model = xgb.train(
    params,
    dtrain,
    num_boost_round=num_rounds,
    evals=evals,
    evals_result=evals_result,
    verbose_eval=1
)

# --- Collect metrics ---
train_logloss = evals_result['train']['mlogloss']
valid_logloss = evals_result['valid']['mlogloss']

# Accuracy at each round
train_acc, valid_acc = [], []
for i in range(1, num_rounds+1):
    y_train_pred = np.argmax(model.predict(dtrain, iteration_range=(0, i)), axis=1)
    y_valid_pred = np.argmax(model.predict(dvalid, iteration_range=(0, i)), axis=1)
    train_acc.append(accuracy_score(y_train, y_train_pred))
    valid_acc.append(accuracy_score(y_valid, y_valid_pred))

    print(f"Round {i}/{num_rounds} "
          f"| Train Loss: {train_logloss[i-1]:.4f}, Train Acc: {train_acc[-1]:.4f} "
          f"| Val Loss: {valid_logloss[i-1]:.4f}, Val Acc: {valid_acc[-1]:.4f}")


[0]	train-mlogloss:1.98312	valid-mlogloss:1.99680
[1]	train-mlogloss:1.81705	valid-mlogloss:1.84221
[2]	train-mlogloss:1.67839	valid-mlogloss:1.71431
[3]	train-mlogloss:1.56142	valid-mlogloss:1.60550
[4]	train-mlogloss:1.46243	valid-mlogloss:1.51360
[5]	train-mlogloss:1.37254	valid-mlogloss:1.42947
[6]	train-mlogloss:1.29227	valid-mlogloss:1.35532
[7]	train-mlogloss:1.22143	valid-mlogloss:1.29000
[8]	train-mlogloss:1.15634	valid-mlogloss:1.23098
[9]	train-mlogloss:1.09820	valid-mlogloss:1.17816
[10]	train-mlogloss:1.04448	valid-mlogloss:1.12824
[11]	train-mlogloss:0.99538	valid-mlogloss:1.08393
[12]	train-mlogloss:0.95038	valid-mlogloss:1.04266
[13]	train-mlogloss:0.90826	valid-mlogloss:1.00383
[14]	train-mlogloss:0.86958	valid-mlogloss:0.96904
[15]	train-mlogloss:0.83313	valid-mlogloss:0.93541
[16]	train-mlogloss:0.79913	valid-mlogloss:0.90441
[17]	train-mlogloss:0.76732	valid-mlogloss:0.87500
[18]	train-mlogloss:0.73805	valid-mlogloss:0.84916
[19]	train-mlogloss:0.71015	valid-mloglos

In [ ]:


import time

# --- Define the path to your test data ---
test_dir = "/content/dataset/Test"

# --- Create the test dataset and dataloader ---
test_dataset = datasets.ImageFolder(test_dir, transform=data_transforms)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# --- Extract and save features for the test set ---
print("Extracting features from the test set...")
start_time = time.time()

X_test, y_test = extract_features(test_loader, feature_extractor, device)

end_time = time.time()
print(f"Feature extraction for test set took: {end_time - start_time:.2f} seconds")

print(f"Test features shape: {X_test.shape}")

# --- Save features for later use ---
np.save('X_test.npy', X_test)
np.save('y_test.npy', y_test)
print("✅ Test features saved!")

Extracting features from the test set...


Extracting features:   1%|▏         | 1/74 [00:00<00:44,  1.63it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Extracting features: 100%|██████████| 74/74 [00:45<00:00,  1.62it/s]

Feature extraction for test set took: 45.75 seconds
Test features shape: (2364, 1664)
✅ Test features saved!


In [ ]:


import numpy as np
import time
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, log_loss

# --- Load the test set features ---
X_test = np.load("X_test.npy")
y_test = np.load("y_test.npy")

# --- Create the DMatrix for the test set ---
dtest = xgb.DMatrix(X_test, label=y_test)

# --- Helper function to calculate TPR and FPR from a confusion matrix ---
def calculate_tpr_fpr(cm):
    """Calculates macro-average TPR and FPR for a multi-class confusion matrix."""
    num_classes = cm.shape[0]
    tpr_list = []
    fpr_list = []

    for i in range(num_classes):
        tp = cm[i, i]
        fn = np.sum(cm[i, :]) - tp
        fp = np.sum(cm[:, i]) - tp
        tn = np.sum(cm) - (tp + fn + fp)

        # True Positive Rate (Recall)
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.
        tpr_list.append(tpr)

        # False Positive Rate
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.
        fpr_list.append(fpr)

    return np.mean(tpr_list), np.mean(fpr_list)


# ===================================================================
#                       PERFORMANCE EVALUATION
# ===================================================================

# Get the class names from the dataset folder structure
class_names = train_dataset.classes

# --- Get predictions (probabilities and final labels) ---
# Training Set
train_start_time = time.time()
y_train_pred_proba = model.predict(dtrain)
train_pred_time = time.time() - train_start_time
y_train_pred = np.argmax(y_train_pred_proba, axis=1)

# Validation Set
valid_start_time = time.time()
y_valid_pred_proba = model.predict(dvalid)
valid_pred_time = time.time() - valid_start_time
y_valid_pred = np.argmax(y_valid_pred_proba, axis=1)

# Testing Set
test_start_time = time.time()
y_test_pred_proba = model.predict(dtest)
test_pred_time = time.time() - test_start_time
y_test_pred = np.argmax(y_test_pred_proba, axis=1)


# --- Calculate Metrics ---
# Training
report_train = classification_report(y_train, y_train_pred, target_names=class_names, output_dict=True)
cm_train = confusion_matrix(y_train, y_train_pred)
tpr_train, fpr_train = calculate_tpr_fpr(cm_train)
loss_train = evals_result['train']['mlogloss'][-1] # Get final loss from training history

# Validation
report_valid = classification_report(y_valid, y_valid_pred, target_names=class_names, output_dict=True)
cm_valid = confusion_matrix(y_valid, y_valid_pred)
tpr_valid, fpr_valid = calculate_tpr_fpr(cm_valid)
loss_valid = evals_result['valid']['mlogloss'][-1] # Get final loss from training history

# Testing
report_test = classification_report(y_test, y_test_pred, target_names=class_names, output_dict=True)
cm_test = confusion_matrix(y_test, y_test_pred)
tpr_test, fpr_test = calculate_tpr_fpr(cm_test)
loss_test = log_loss(y_test, y_test_pred_proba) # Calculate test loss manually
auroc_test = roc_auc_score(y_test, y_test_pred_proba, multi_class='ovr', average='weighted')


# --- Print Final Reports ---

print("="*50)
print("          TRAINING SET METRICS")
print("="*50)
print(f"Accuracy:          {report_train['accuracy']:.4f}")
print(f"Weighted Precision:  {report_train['weighted avg']['precision']:.4f}")
print(f"Weighted Recall/TPR: {report_train['weighted avg']['recall']:.4f}")
print(f"Weighted F1-Score:   {report_train['weighted avg']['f1-score']:.4f}")
print(f"Macro Avg FPR:       {fpr_train:.4f}")
print(f"Final Loss:          {loss_train:.4f}")
# Note: Training time is the time for the entire xgb.train() call
print(f"Inference Time:      {train_pred_time:.4f} seconds\n")


print("="*50)
print("         VALIDATION SET METRICS")
print("="*50)
print(f"Accuracy:          {report_valid['accuracy']:.4f}")
print(f"Weighted Precision:  {report_valid['weighted avg']['precision']:.4f}")
print(f"Weighted Recall/TPR: {report_valid['weighted avg']['recall']:.4f}")
print(f"Weighted F1-Score:   {report_valid['weighted avg']['f1-score']:.4f}")
print(f"Macro Avg FPR:       {fpr_valid:.4f}")
print(f"Final Loss:          {loss_valid:.4f}")
print(f"Validation Time:     {valid_pred_time:.4f} seconds\n")


print("="*50)
print("           TESTING SET METRICS")
print("="*50)
print(f"Accuracy:          {report_test['accuracy']:.4f}")
print(f"Weighted Precision:  {report_test['weighted avg']['precision']:.4f}")
print(f"Weighted Recall/TPR: {report_test['weighted avg']['recall']:.4f}")
print(f"Weighted F1-Score:   {report_test['weighted avg']['f1-score']:.4f}")
print(f"Macro Avg FPR:       {fpr_test:.4f}")
print(f"Weighted AUROC:      {auroc_test:.4f}")
print(f"Final Loss:          {loss_test:.4f}")
print(f"Testing Time:        {test_pred_time:.4f} seconds\n")

          TRAINING SET METRICS
Accuracy:          0.9999
Weighted Precision:  0.9999
Weighted Recall/TPR: 0.9999
Weighted F1-Score:   0.9999
Macro Avg FPR:       0.0000
Final Loss:          0.0211
Inference Time:      0.0023 seconds

         VALIDATION SET METRICS
Accuracy:          0.9094
Weighted Precision:  0.9093
Weighted Recall/TPR: 0.9094
Weighted F1-Score:   0.9093
Macro Avg FPR:       0.0113
Final Loss:          0.2850
Validation Time:     0.0005 seconds

           TESTING SET METRICS
Accuracy:          0.8993
Weighted Precision:  0.8995
Weighted Recall/TPR: 0.8993
Weighted F1-Score:   0.8993
Macro Avg FPR:       0.0126
Weighted AUROC:      0.9927
Final Loss:          0.3109
Testing Time:        0.1137 seconds

